# 📷 Google Photos Takeout 雲端自動化增量整理工坊 (高視覺即時進度版)

本筆記本專為 **Google One 2TB** 雲端生態設計，直接結合 `14_Google-Photos-Takeout-Organizer` 官方核心引擎。

### 核心亮點：
1. **智慧批次過濾**：自動鎖定最新一波 Takeout 匯出，並自動排除重複下載的 `(1).zip` 副本與檢查缺卷。
2. **即時百分比進度顯示**：解壓、配對、歸檔、驗證每一步都有清晰進度條與檔案計數，不再盲目等待！
3. **全自動掛載 Google Drive**：直接在 Google 雲端內部高速處理，本機電腦 0 下載、0 流量。
4. **全域 Sidecar 雜湊配對 & 五層日期保底**：秒級解析數萬筆 JSON 時間戳記，精準校正相片建立時間。
5. **iPhone Live Photo 綁定**：`.HEIC` 與 `.MOV` 同步歸檔，動態照片不拆散。
6. **SHA-256 增量去重與安全斷點**：支援同卷斷點續跑與跨次驗證，驗證 100% 寫入無誤後才清理暫存區。

## 步驟 1：掛載 Google 雲端硬碟
點擊左側播放鍵 ▶️，依提示授權存取 Google Drive：

In [ ]:
from google.colab import drive
import os, sys, shutil, glob, time, re
from pathlib import Path

# 1. 鎖定台灣標準時區 (Asia/Taipei, UTC+8)，確保時間判定與本機完全一致
!ln -snf /usr/share/zoneinfo/Asia/Taipei /etc/localtime && echo "Asia/Taipei" > /etc/timezone
os.environ['TZ'] = 'Asia/Taipei'
try:
    time.tzset()
except AttributeError:
    pass

drive.mount('/content/drive')
print('✓ Google Drive 掛載成功且時區已鎖定為台灣標準時間 (Asia/Taipei)！')

## 步驟 2：安裝 14 號專案核心引擎與相依套件

In [ ]:
# 先安裝影像解碼相依套件，再以 --no-deps 極速安裝最新核心引擎（免下載桌面版 PySide6）
!pip install -q pillow pillow-heif
!pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/lianghao02/14_Google-Photos-Takeout-Organizer.git
print('✓ 核心整理引擎最新版本安裝完成！')


## 步驟 3：設定路徑、智慧鎖定最新匯出批次並排除重複 `(1).zip`

In [ ]:
from collections import defaultdict

DRIVE_ROOT = Path('/content/drive/MyDrive')
TAKEOUT_DIR = DRIVE_ROOT / 'Takeout'
OUTPUT_DIR = DRIVE_ROOT / 'Organized_Photos'

# 預設僅處理最新一波 Takeout 匯出，並自動略過重複的 (1).zip 副本
ONLY_LATEST_BATCH = True
SKIP_DUPLICATE_COPIES = True

TAKEOUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_zip_files = sorted(list(TAKEOUT_DIR.glob('*.zip')))
zip_pat = re.compile(r'^takeout-(\d{8}T\d{6}Z)-(?:(\d+)-)?(\d{3})(?:\s*\((\d+)\))?\.zip$', re.IGNORECASE)

batches = defaultdict(list)
other_zips = []
for z in all_zip_files:
    m = zip_pat.match(z.name)
    if m:
        batch_id, part_grp, vol_idx, copy_num = m.groups()
        batches[batch_id].append((z, part_grp, int(vol_idx), copy_num))
    else:
        other_zips.append(z)

skipped_old_zips = []
skipped_copy_zips = []
zip_files = []

if batches:
    sorted_batch_ids = sorted(batches.keys())
    target_batch_ids = [sorted_batch_ids[-1]] if ONLY_LATEST_BATCH else sorted_batch_ids
    if ONLY_LATEST_BATCH and len(sorted_batch_ids) > 1:
        for old_id in sorted_batch_ids[:-1]:
            skipped_old_zips.extend(item[0] for item in batches[old_id])
        print(f'ℹ️ 偵測到多個匯出批次 {sorted_batch_ids}，已自動鎖定最新批次：{target_batch_ids[0]}（略過舊批次 {len(skipped_old_zips)} 檔）')

    for b_id in target_batch_ids:
        items = batches[b_id]
        by_vol = defaultdict(list)
        for item in items:
            by_vol[(item[1], item[2])].append(item)
        vol_numbers = sorted(k[1] for k in by_vol.keys() if k[0] is not None)
        if vol_numbers:
            missing_vols = sorted(set(range(1, max(vol_numbers) + 1)) - set(vol_numbers))
            if missing_vols:
                print(f'⚠️ 注意：批次 {b_id} 缺少分卷編號：{missing_vols}')
        for key in sorted(by_vol.keys(), key=lambda x: (x[0] or '', x[1])):
            candidates = sorted(by_vol[key], key=lambda x: (x[3] is not None, int(x[3] or 0)))
            if SKIP_DUPLICATE_COPIES and len(candidates) > 1:
                zip_files.append(candidates[0][0])
                skipped_copy_zips.extend(c[0] for c in candidates[1:])
            else:
                zip_files.extend(c[0] for c in candidates)

zip_files.extend(other_zips)
if skipped_copy_zips:
    saved_gb = sum(z.stat().st_size for z in skipped_copy_zips) / (1024 ** 3)
    print(f'ℹ️ 已自動排除同批次重複 (1).zip 副本：{len(skipped_copy_zips)} 檔（省下 {saved_gb:.1f} GB 重複解壓）')

total_size_gb = sum(z.stat().st_size for z in zip_files) / (1024 ** 3)

# 智慧容量判定：檢查 Colab 本機磁碟 (/content) 剩餘空間
colab_total, colab_used, colab_free = shutil.disk_usage('/content')
colab_free_gb = colab_free / (1024 ** 3)
needed_gb = (total_size_gb * 1.2) + 15

if zip_files and colab_free_gb >= needed_gb:
    WORK_DIR = Path('/content/Takeout_Work')
    work_mode = '⚡ Colab 本地高速 SSD (極速處理模式)'
else:
    WORK_DIR = DRIVE_ROOT / 'Takeout_Work'
    work_mode = '💾 Google Drive 雲端空間 (大容量安全保底模式)'

WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f'相簿目標目錄：{OUTPUT_DIR}')
print(f'本次實際處理 Takeout 壓縮檔：{len(zip_files)} 卷 (總計約 {total_size_gb:.1f} GB)')
print(f'Colab 本地可用空間：{colab_free_gb:.1f} GB / 估計需空間：{needed_gb:.1f} GB')
print(f'工作暫存區決策：{work_mode}')
print(f'實際暫存路徑：{WORK_DIR}')

for idx, z in enumerate(zip_files[:5], 1):
    size_mb = z.stat().st_size / (1024 * 1024)
    print(f'  [{idx:03d}] {z.name} ({size_mb:.1f} MB)')
if len(zip_files) > 5:
    print(f'  ... 略過中間 {len(zip_files)-5} 個檔案 ...')
    last_z = zip_files[-1]
    print(f'  [{len(zip_files):03d}] {last_z.name} ({last_z.stat().st_size / (1024*1024):.1f} MB)')

if not zip_files:
    print('\n⚠️ 目前 Takeout 目錄下沒有 ZIP 檔案。若 Google 尚未完成打包，請稍候再試。')

## 步驟 4：執行分析、規劃、去重與無損匯出 (含即時進度顯示)

In [ ]:
# 強制清除 Python 模組記憶體快取，確保載入剛安裝的最新極速版引擎
import sys
for mod in list(sys.modules.keys()):
    if 'google_photos_takeout_organizer' in mod:
        del sys.modules[mod]

if not zip_files:
    raise RuntimeError('沒有待處理的 ZIP 檔案，流程終止。')

import time
from google_photos_takeout_organizer.service import analyze
from google_photos_takeout_organizer.exporter import export
from google_photos_takeout_organizer.verifier import verify

# 定義即時進度回呼函式
last_update = [0]
def console_progress(stage, current, total, message):
    now = time.time()
    pct = (current / total * 100) if total else 0
    if now - last_update[0] > 0.3 or current == total or current == 1:
        last_update[0] = now
        stage_labels = {
            'ANALYZE_EXTRACT': '📦 解壓封存檔',
            'ANALYZE': '🔎 分析配對中',
            'EXPORT': '📁 歸檔寫入中',
            'VERIFY': '🛡️ 完整性驗證',
        }
        prefix = stage_labels.get(stage, '⏳ 處理中')
        sys.stdout.write(f'\r{prefix} [{pct:5.1f}%] ({current}/{total}) - {message[:50]}')
        sys.stdout.flush()
        if current == total:
            sys.stdout.write('\n')

print('====================================================')
print('1/3 階段：解壓 ZIP、全域 Sidecar 配對與五層日期校準...')
print('====================================================')
manifest = analyze(zip_files, WORK_DIR, progress_callback=console_progress)
manifest_path = WORK_DIR / 'manifest.json'
print('\n✓ 1/3 階段分析完成！摘要：')
for k, v in manifest['summary'].items():
    print(f'  - {k}: {v}')

print('\n====================================================')
print('2/3 階段：執行無損增量歸檔 (去重複與年月分類)...')
print('====================================================')
export_result = export(manifest_path, OUTPUT_DIR, progress_callback=console_progress)
print('\n✓ 2/3 階段歸檔完成！摘要：')
for k, v in export_result['summary'].items():
    print(f'  - {k}: {v}')

print('\n====================================================')
print('3/3 階段：校驗匯出成果 (SHA-256 與位元組完整性)...')
print('====================================================')
verify_result = verify(manifest_path, OUTPUT_DIR, progress_callback=console_progress)
print('\n驗證結果：')
print(verify_result)

is_verified_ok = verify_result.get('result') == 'PASS' or verify_result.get('status') == 'OK'
if is_verified_ok:
    print('\n🎉 所有相片與影片 100% 完整無誤寫入！')
else:
    print('\n⚠️ 部分檔案校驗未通過，請查看上方訊息。')

## 步驟 5：安全清理 Takeout 壓縮檔與暫存區 (釋放雲端硬碟空間)
**注意**：僅在校驗 100% 通過 (`result == 'PASS'`) 時才會刪除原始 ZIP。

In [ ]:
# 設定是否刪除已處理的 ZIP 檔案與重複/舊批次 ZIP (預設為 True 釋放空間)
DELETE_SOURCE_ZIPS = True
DELETE_SKIPPED_OLD_AND_COPY_ZIPS = True

is_verified_ok = verify_result.get('result') == 'PASS' or verify_result.get('status') == 'OK'
if is_verified_ok:
    if DELETE_SOURCE_ZIPS:
        targets = list(zip_files)
        if DELETE_SKIPPED_OLD_AND_COPY_ZIPS:
            targets.extend(skipped_copy_zips)
            targets.extend(skipped_old_zips)
        for z in targets:
            try:
                if z.exists():
                    z.unlink()
                    print(f'✓ 已安全刪除封存檔：{z.name}')
            except Exception as e:
                print(f'刪除失敗 {z.name}: {e}')

    # 清理工作目錄中的暫存 extracted 檔案，保留 manifest 與 report
    extracted_dir = WORK_DIR / 'extracted'
    if extracted_dir.exists():
        shutil.rmtree(extracted_dir, ignore_errors=True)
        print(f'✓ 已清理暫存目錄：{extracted_dir}')

    print('\n✨ 任務圓滿完成！雲端硬碟空間已釋放，相片已妥善歸檔至 Organized_Photos/')
else:
    print('❌ 因驗證未完全通過，已嚴格保留所有原始 ZIP 檔案與暫存區，確保資料絕對安全！')